In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from cellina import make_neighbor_perturbation
from cellina_graph import make_perturbed_expression
from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from counterfactual_analysis import get_perturbation_logfc, get_global_perturbation_logfc
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import cellina
import cellina_graph

cellina.__version__, cellina_graph.__version__

('0.7.4', '0.0.11')

In [5]:
set_seed(0)

In [6]:
DATASET_NAME = "crc"  # or "merfish"
CELLINA_BASE_MODEL_ROOT = os.path.join('/data/a330d', "data/ood/trained")
CELLINA_GAT_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")

In [7]:
CRC_PATHS = [
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_120.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_221.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [8]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 512
library_size = 'latent'
n_deg = 50
n_pert_genes = 200

In [9]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".h5ad")[0] for path in PATHS]

In [10]:
results = []
model_names = ['cellina-graph-W'] #['cellina-W']#, 

for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    
    for holdout_celltype in HOLDOUT_CELLTYPES:
        # 50 times * in print
        print(f"{'='*50} Slide: {slide_id}, Holdout Celltype: {holdout_celltype} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_celltype,
                                                    labels_key=labels_key,
                                                    domains_key=domains_key,
                                                    holdout_domains=holdout_domains,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        # Compute spatial features after splitting to avoid data leakage
        step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
        adata = preprocess_spatial_features(adata, step_size_px=step_size_px, n_neighbors=n_neighbors, test_indices=test_idx)
        
        for model_name in model_names:
            if model_name == 'cellina-W':
                 save_name = "cellina-pert"
                 model_class = "cellina"
            else:
                save_name = "cellina-gat-pert"
                model_class = "cellina_graph"
            
                        
            MODEL_ROOT = CELLINA_BASE_MODEL_ROOT if model_class == 'cellina' else CELLINA_GAT_MODEL_ROOT
            save_dir = os.path.join(MODEL_ROOT, slide_id, holdout_celltype, model_name)
            
            try:
                model = _load_model(save_dir,
                                    model_class=model_class,
                                    adata=adata,
                                    splits=splits)
            except Exception as e:
                print(f"Failed to load model from {save_dir} with error: {e}")
                continue
            is_control_region = adata.obs[domains_key]==(control_domain)
            is_holdout_ct = adata.obs[labels_key].astype(str) == holdout_celltype
            mask_control = is_control_region & is_holdout_ct
            idx_control = np.where(mask_control.values)[0]    
            
            # Only for cellina-base, cellina-gat will reconvert to raw counts
            if model_class == 'cellina':
                adata.X = adata.layers['counts'].copy()
                sc.pp.normalize_total(adata, target_sum=COUNTS_PER_K)
                sc.pp.log1p(adata)
            else:
                adata.X = adata.layers['counts'].copy()
            for hd in holdout_domains:
                # ── Cell-type-specific perturbation ──────────────────────────────────────
                domain_logfc_df = get_perturbation_logfc(adata, control_domain, hd, labels_key, domains_key)
                global_logfc_series = get_global_perturbation_logfc(adata, control_domain, hd, labels_key, domains_key, holdout_celltype)
                domain_logfc_df.loc[holdout_celltype, global_logfc_series.index] = global_logfc_series # NOTE: set the holdout celltype's perturbation to the global perturbation - ct

                logfc_series_dict = {}
                for ct in domain_logfc_df.index:
                    s = domain_logfc_df.loc[ct]
                    top_g = s.abs().nlargest(n_pert_genes).index.tolist()
                    logfc_series_dict[ct] = s[top_g]

                if model_class == 'cellina':
                    make_neighbor_perturbation(
                            adata,
                            perturbations=logfc_series_dict,
                            groupby=labels_key,
                            obsm_key_out='spatial_x_cf',
                            base=np.e,
                            renormalize=True,
                            add_shift=True,
                    )
                    pert_expr = model.get_perturbed_expression(
                            adata=adata, indices=idx_control, spatial_obsm_key='spatial_x_cf',
                            batch_size=batch_size, library_size=library_size,
                            )
                else:
                    cf_layer_key = f'counts_cf_{hd.lower()}'
                    make_perturbed_expression(
                        adata,
                        perturbations=logfc_series_dict,
                        groupby=labels_key,
                        layer_key=cf_layer_key,
                        base=np.e,
                        add_shift=False,
                        renormalize=True)
                    pert_expr = model.get_perturbed_expression(
                        adata=adata, indices=idx_control,
                        cf_layer=cf_layer_key,
                        batch_size=batch_size, library_size=library_size,
                        )
                
                is_holdout_region = adata.obs[domains_key].astype(str) == hd
                mask_target = is_holdout_region & is_holdout_ct
                idx_target = np.where(mask_target.values)[0]
                
                # Compute stats
                control = adata.layers['counts'][mask_control.values, :]
                target = adata.layers['counts'][mask_target.values, :]
                control, target = _to_dense(control), _to_dense(target)
                counterfactual = pert_expr

                gt_lfc, cf_lfc, deg = get_lfc(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)

                spear, _ = spearmanr(gt_lfc[deg], cf_lfc[deg])
                pear, _ = pearsonr(gt_lfc[deg], cf_lfc[deg])
                prec = precision(gt_lfc, cf_lfc, k=n_deg, use_abs=True)
                dir_match = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="intersection")
                dir_match_k = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="k")
                dir_match_gt = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="gt_topk")
                mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=COUNTS_PER_K)
                edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K)
                edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True)
                edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True)
                edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True, log1p=False)
                rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg, library_size=COUNTS_PER_K)
                mse_lfc = compute_mse_lfc(gt_vec=gt_lfc, cf_vec=cf_lfc, deg=deg)

                results.append(
                        dict(
                        dataset_name=DATASET_NAME,
                        sid=slide_id,
                        control_domain=control_domain,
                        target_domain=hd,
                        n_deg=n_deg,
                        model_name=save_name,
                        holdout_celltype=holdout_celltype,
                        spearman=spear,
                        pearson=pear,
                        precision=prec,
                        direction_match=dir_match,
                        direction_match_k=dir_match_k,
                        direction_match_gt=dir_match_gt,
                        mixing_index=mix_idx,
                        edistance_global=edist_global,
                        edistance_local=edist_local,
                        edistance_pca_log=edist_pca_log,
                        edistance_pca=edist_pca,
                        rmse=rmse,
                        mse_lfc=mse_lfc,
                        top_n_perturb=n_pert_genes,
                        )
                )

/data/a330d/projects/cellina-reproducibility/notebooks/loo_benchmarks/../../scripts/train_loo.py:175: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[labels_key] = adata.obs[labels_key].astype("category")


================================================== Slide: crc_120, Holdout Celltype: Endothelial ==================================================


INFO: Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
2026-06-04 11:00:55 | [INFO] Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data2/a330d/data/ood/trained/crc_120/Endothelial/cellina-graph-W/model.pt already downloaded        


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


...

INFO: Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
2026-06-04 12:28:37 | [INFO] Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data2/a330d/data/ood/trained/crc_242/T_cell/cellina-graph-W/model.pt already downloaded             


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting with edge prediction   
cellina_graph loaded model from /data2/a330d/data/ood/trained/crc_242/T_cell/cellina-graph-W


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: The usage of the 'directed' argument in 'NeighborSampler' is deprecated. Use `subgraph_type='induced'` instead.
  neighbor_sampler = NeighborSampler(


In [15]:
results_df = pd.DataFrame(results)
results_df['rmse'] = np.log10(results_df['rmse'])
results_df['rmse_lfc'] = np.sqrt(results_df['mse_lfc'])

In [16]:
metrics = ['spearman', 'pearson', 'direction_match_k', 'edistance_pca_log', 'rmse', 'rmse_lfc']
for metric in metrics:
    mean_value = results_df[metric].mean()
    print(f"{metric}:", mean_value)

spearman: 0.7007410964385752
pearson: 0.81829673
direction_match_k: 0.46066666666666667
edistance_pca_log: 8.157589127222696
rmse: 3.633079
rmse_lfc: 6.3774314


In [17]:
old_df = pd.read_csv('../../results/loo_summary_merfish_DEG_50.csv')
metrics = ['spearman', 'pearson', 'signed_precision', 'e-distance', 'rmse', 'rmse_lfc']

old_df = old_df[old_df["model_name"]=='cellina-pert']
for metric in metrics:
    mean_value = old_df[metric].mean()
    print(f"{metric}:", mean_value)

spearman: 0.7094005602240895
pearson: 0.8270286783333333
signed_precision: 0.484
e-distance: 9.268833894729616
rmse: 3.6296088973802028
rmse_lfc: 6.296049240970972


In [11]:
results_csv_name = f'../../results/loo_cellina_{DATASET_NAME}_DEG_{n_deg}'
results_csv_path = results_csv_name + '_pert_v2.csv'
df_results = pd.DataFrame(results)

# If file exists, append results to existing csv, otherwise create new csv
if os.path.exists(results_csv_path):
    df_results.to_csv(f"{results_csv_path}", index=False, mode='a', header=False)
else:
    df_results.to_csv(f"{results_csv_path}", index=False)

In [13]:
df_results = pd.read_csv(results_csv_path)

In [17]:
# Take mean of metrics spearman, pearson, direction_match_k, edistance_pca_log
summary_df = df_results.groupby(['dataset_name', 'model_name']).agg({
    'spearman': 'mean',
    'pearson': 'mean',
    'direction_match_k': 'mean',
    'edistance_pca_log': 'mean'
}).reset_index()
summary_df


,dataset_name,model_name,spearman,pearson,direction_match_k,edistance_pca_log
0,crc,cellina-gat-pert,0.66110,0.733897,0.321333,8.324474
1,crc,cellina-pert,0.73714,0.853546,0.410667,7.816848


In [ ]:
metrics = ['spearman', 'pearson', 'direction_match_k', 'edistance_pca_log', 'rmse', 'rmse_lfc']
for metric in metrics:
    mean_value = results_df[metric].mean()
    print(f"{metric}:", mean_value)